# Recent Results Overview

This notebook focuses on the most recent probe result group under `results/`, with Pearson correlation as the primary evaluation metric.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

plt.style.use("ggplot")
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200


In [ ]:
ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "results"

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

In [ ]:
EVAL_ADOPTION_PROBE_RE = re.compile(
    r"^(?:eval-adoption-)?absolute_accuracy_decay__(?P<perturbation_type>.+?)__(?:(?P<probe_control>none|randomization|permutation)__)?L(?P<layer>\d{3})$"
)

def parse_run_path(path: Path) -> dict:
    rel = path.relative_to(RESULTS_DIR)
    parts = rel.parts
    if len(parts) < 8:
        raise ValueError(f"Unexpected results path: {rel}")

    probe_idx = next((i for i, part in enumerate(parts) if EVAL_ADOPTION_PROBE_RE.match(part) or re.search(r"(?:_L|__L)\d{3}$", part)), None)
    if probe_idx is None:
        raise ValueError(f"Could not locate probe segment in {rel}")

    probe_name = parts[probe_idx]
    tail = parts[probe_idx + 1 :]
    if len(tail) == 7:
        model_name, encoding, control_task, sample_size, seed, fold, filename = tail
        status = "completed"
        version = ""
    elif len(tail) == 8:
        model_name, encoding, control_task, sample_size, seed, fold, status, filename = tail
        version = ""
    elif len(tail) == 9:
        model_name, encoding, control_task, sample_size, seed, fold, status, version, filename = tail
    else:
        raise ValueError(f"Unexpected run tail after probe {probe_name}: {tail}")

    layer_match = re.search(r"(?:_L|__L)(\d{3})", probe_name)
    if layer_match is None:
        raise ValueError(f"Could not parse layer from {probe_name}")

    perturbation_match = EVAL_ADOPTION_PROBE_RE.match(probe_name)
    perturbation_type = perturbation_match.group("perturbation_type") if perturbation_match else None

    return {
        "results_group": "/".join(parts[:probe_idx]),
        "probe_name": probe_name,
        "layer": int(layer_match.group(1)),
        "model_name": model_name,
        "encoding": encoding,
        "control_task": control_task,
        "perturbation_type": perturbation_type,
        "sample_size": int(sample_size),
        "fold": int(fold),
        "seed": int(seed),
        "status": status,
        "version": version,
        "filename": filename,
        "path": path,
        "mtime": path.stat().st_mtime,
    }


def infer_metadata_path(results_group: str) -> Path | None:
    if "eval_adoption_absolute_accuracy_decay_aggregated" in results_group:
        candidate = ROOT / "data" / "eval_adoption_internals_aggregated" / "metadata.csv"
    elif "eval_adoption_absolute_accuracy_decay_table" in results_group:
        candidate = ROOT / "data" / "eval_adoption_internals_table" / "metadata.csv"
    else:
        candidate = ROOT / "data" / "internals" / "metadata.csv"
    return candidate if candidate.exists() else None


def last_non_null(series: pd.Series):
    cleaned = series.dropna()
    return cleaned.iloc[-1] if not cleaned.empty else np.nan


def maybe_metric(df: pd.DataFrame, column: str, reducer: str = "last"):
    if column not in df.columns:
        return np.nan
    series = df[column]
    if reducer == "last":
        return last_non_null(series)
    if reducer == "max":
        return series.max(skipna=True)
    if reducer == "min":
        return series.min(skipna=True)
    raise ValueError(reducer)


def load_metrics() -> tuple[pd.DataFrame, pd.DataFrame]:
    run_rows = []
    curve_frames = []

    for metrics_path in sorted(RESULTS_DIR.rglob("metrics.csv")):
        info = parse_run_path(metrics_path)
        df = pd.read_csv(metrics_path)

        curve = df.copy()
        for key, value in info.items():
            if key not in {"path", "filename"}:
                curve[key] = value
        curve_frames.append(curve)

        run_rows.append({
            **{k: v for k, v in info.items() if k not in {"path", "filename"}},
            "metrics_path": str(metrics_path),
            "epochs_logged": len(df),
            "best_val_loss": maybe_metric(df, "val loss", "min"),
            "best_val_error": maybe_metric(df, "val error", "min"),
            "best_val_pearson": maybe_metric(df, "val pearson", "max"),
            "full_test_error": maybe_metric(df, "full test error", "last"),
            "full_test_pearson": maybe_metric(df, "full test pearson", "last"),
        })

    runs_df = pd.DataFrame(run_rows)
    runs_df = runs_df[~runs_df["results_group"].str.contains("pca50", case=False, na=False)].copy()
    runs_df = runs_df.sort_values(["results_group", "layer", "control_task"]).reset_index(drop=True)
    curves_df = pd.concat(curve_frames, ignore_index=True)
    curves_df = curves_df[~curves_df["results_group"].str.contains("pca50", case=False, na=False)].copy()
    curves_df = curves_df.sort_values(["results_group", "layer", "control_task", "step"])
    return runs_df, curves_df


def load_predictions() -> pd.DataFrame:
    pred_frames = []

    for preds_path in sorted(RESULTS_DIR.rglob("preds.csv")):
        info = parse_run_path(preds_path)
        df = pd.read_csv(preds_path)
        df = df.rename(columns={df.columns[0]: "row_idx"})
        df["problem_id"] = df["instance"].str.extract(r"problem_(\d+)")[0].astype("string")
        df["row_id"] = df["instance"].str.extract(r"row_(\d+)")[0].astype("string")
        df["pred_abs_error"] = (df["pred"] - df["label"]).abs()
        df["pred_is_exact"] = np.isclose(df["pred"], df["label"]).astype(int)
        for key, value in info.items():
            if key not in {"path", "filename"}:
                df[key] = value

        metadata_path = infer_metadata_path(info["results_group"])
        if metadata_path is not None:
            metadata = pd.read_csv(metadata_path)
            merge_key = "row_id" if "row_id" in metadata.columns and df["row_id"].notna().any() else "problem_id"
            if merge_key in metadata.columns:
                metadata = metadata.copy()
                metadata[merge_key] = metadata[merge_key].astype("string")
                df[merge_key] = df[merge_key].astype("string")
                df = df.merge(metadata, on=merge_key, how="left", suffixes=("", "_meta"))

        pred_frames.append(df)

    return pd.concat(pred_frames, ignore_index=True).sort_values(["results_group", "layer", "control_task"]).reset_index(drop=True)


runs_df, curves_df = load_metrics()


In [ ]:
latest_group = (
    runs_df.groupby("results_group")["mtime"]
    .max()
    .sort_values(ascending=False)
    .index[0]
)

latest_runs = runs_df[runs_df["results_group"] == latest_group].copy()
latest_curves = curves_df[curves_df["results_group"] == latest_group].copy()
latest_preds = preds_df[preds_df["results_group"] == latest_group].copy()

latest_is_eval_adoption = "eval_adoption" in latest_group
allowed_controls = ["NONE", "RANDOMIZATION"]

if latest_is_eval_adoption:
    latest_runs = latest_runs[latest_runs["control_task"].isin(allowed_controls)].copy()
    latest_curves = latest_curves[latest_curves["control_task"].isin(allowed_controls)].copy()
    latest_preds = latest_preds[latest_preds["control_task"].isin(allowed_controls)].copy()
    latest_runs["probe_control"] = latest_runs["control_task"].str.lower()
    latest_curves["probe_control"] = latest_curves["control_task"].str.lower()
    latest_preds["probe_control"] = latest_preds["control_task"].str.lower()
    latest_runs["analysis_label"] = latest_runs["perturbation_type"] + " | " + latest_runs["probe_control"]
    latest_curves["analysis_label"] = latest_curves["perturbation_type"] + " | " + latest_curves["probe_control"]
    latest_preds["analysis_label"] = latest_preds["perturbation_type"] + " | " + latest_preds["probe_control"]
else:
    latest_runs["analysis_label"] = latest_runs["control_task"]
    latest_curves["analysis_label"] = latest_curves["control_task"]
    latest_preds["analysis_label"] = latest_preds["control_task"]

pd.DataFrame({
    "latest_results_group": [latest_group],
    "is_eval_adoption": [latest_is_eval_adoption],
    "n_runs": [len(latest_runs)],
    "n_prediction_rows": [len(latest_preds)],
    "controls": [", ".join(sorted(latest_runs["control_task"].dropna().unique()))],
    "perturbation_types": [", ".join(sorted(latest_runs["perturbation_type"].dropna().unique())) if latest_is_eval_adoption else "n/a"],
    "layers": [latest_runs["layer"].nunique()],
})


In [ ]:
if latest_is_eval_adoption:
    pearson_summary = latest_runs.groupby(["probe_control", "perturbation_type"]).agg(
        best_full_test_pearson=("full_test_pearson", "max"),
        mean_full_test_pearson=("full_test_pearson", "mean"),
        min_full_test_error=("full_test_error", "min"),
    ).round(4).reset_index().sort_values(["probe_control", "best_full_test_pearson"], ascending=[True, False])
else:
    pearson_summary = latest_runs.groupby("analysis_label").agg(
        best_full_test_pearson=("full_test_pearson", "max"),
        mean_full_test_pearson=("full_test_pearson", "mean"),
        min_full_test_error=("full_test_error", "min"),
    ).round(4).sort_values("best_full_test_pearson", ascending=False)
pearson_summary


In [ ]:
if latest_is_eval_adoption:
    best_layers = (
        latest_runs.sort_values(["probe_control", "perturbation_type", "full_test_pearson"], ascending=[True, True, False])
        .groupby(["probe_control", "perturbation_type"])
        .head(1)
        [["probe_control", "perturbation_type", "layer", "full_test_pearson", "full_test_error"]]
        .sort_values(["probe_control", "full_test_pearson"], ascending=[True, False])
        .reset_index(drop=True)
    )
else:
    best_layers = (
        latest_runs.sort_values(["analysis_label", "full_test_pearson"], ascending=[True, False])
        .groupby("analysis_label")
        .head(1)
        [["analysis_label", "layer", "full_test_pearson", "full_test_error"]]
        .sort_values("full_test_pearson", ascending=False)
    )
best_layers


In [ ]:
if latest_is_eval_adoption:
    fig = px.line(
        latest_runs.sort_values("layer"),
        x="layer",
        y="full_test_pearson",
        color="perturbation_type",
        facet_col="probe_control",
        markers=True,
        category_orders={"probe_control": ["none", "randomization"]},
        title="Full-Test Pearson by Layer, Control Task, and Perturbation Type",
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.replace("probe_control=", "control=")))
    fig.update_xaxes(title_text="Layer")
    fig.update_yaxes(title_text="Pearson correlation")
    fig.show()
else:
    fig, ax = plt.subplots(figsize=(14, 5))
    for analysis_label, group in latest_runs.groupby("analysis_label"):
        group = group.sort_values("layer")
        ax.plot(group["layer"], group["full_test_pearson"], marker="o", linewidth=2, label=analysis_label)
    ax.set_title("Full-Test Pearson by Layer")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Pearson correlation")
    ax.legend(title="analysis_label")
    plt.tight_layout()


In [ ]:
if latest_is_eval_adoption:
    fig = px.scatter(
        latest_runs,
        x="layer",
        y="perturbation_type",
        color="full_test_pearson",
        facet_col="probe_control",
        category_orders={"probe_control": ["none", "randomization"]},
        color_continuous_scale="RdBu",
        range_color=[-1, 1],
        title="Full-Test Pearson Heatmap View",
    )
    fig.update_traces(marker={"size": 14, "symbol": "square"})
    fig.for_each_annotation(lambda a: a.update(text=a.text.replace("probe_control=", "control=")))
    fig.update_xaxes(title_text="Layer")
    fig.update_yaxes(title_text="Perturbation type")
    fig.show()
else:
    heatmap_df = latest_runs.pivot(index="analysis_label", columns="layer", values="full_test_pearson").sort_index()
    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(heatmap_df.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title("Full-Test Pearson Heatmap")
    ax.set_xlabel("Layer")
    ax.set_ylabel("analysis_label")
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns)
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)
    plt.colorbar(im, ax=ax, label="Pearson correlation")
    plt.tight_layout()


In [ ]:
if latest_is_eval_adoption:
    fig = px.line(
        latest_runs.sort_values("layer"),
        x="layer",
        y="full_test_error",
        color="perturbation_type",
        facet_col="probe_control",
        markers=True,
        category_orders={"probe_control": ["none", "randomization"]},
        title="Full-Test Error by Layer, Control Task, and Perturbation Type",
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.replace("probe_control=", "control=")))
    fig.update_yaxes(title_text="Full-test error")
    fig.update_xaxes(title_text="Layer")
    fig.show()
else:
    fig, ax = plt.subplots(figsize=(14, 5))
    for analysis_label, group in latest_runs.groupby("analysis_label"):
        group = group.sort_values("layer")
        ax.plot(group["layer"], group["full_test_error"], marker="o", linewidth=2, label=analysis_label)
    ax.set_title("Supporting View: Full-Test Error by Layer")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Error")
    ax.legend(title="analysis_label")
    plt.tight_layout()


In [ ]:
if latest_is_eval_adoption:
    top_pairs = best_layers[["probe_control", "perturbation_type", "layer"]].head(6)
    pair_frames = []
    for row in top_pairs.itertuples(index=False):
        subset = latest_preds[(latest_preds["probe_control"] == row.probe_control) & (latest_preds["perturbation_type"] == row.perturbation_type) & (latest_preds["layer"] == row.layer)].copy()
        subset["facet_label"] = f"{row.perturbation_type} | {row.probe_control} | L{row.layer:03d}"
        pair_frames.append(subset)
    scatter_df = pd.concat(pair_frames, ignore_index=True) if pair_frames else pd.DataFrame()
    if not scatter_df.empty:
        fig = px.scatter(
            scatter_df,
            x="label",
            y="pred",
            color="perturbation_type",
            facet_col="probe_control",
            facet_row="facet_label",
            opacity=0.7,
            category_orders={"probe_control": ["none", "randomization"]},
            title="Prediction vs Label for Best Layers",
        )
        fig.for_each_annotation(lambda a: a.update(text=a.text.replace("probe_control=", "control=").replace("facet_label=", "pair=")))
        fig.update_xaxes(title_text="True label")
        fig.update_yaxes(title_text="Prediction")
        fig.show()
else:
    top_labels = best_layers["analysis_label"].tolist()[:4]
    fig, axes = plt.subplots(len(top_labels), 1, figsize=(8, 4 * max(1, len(top_labels))))
    axes = np.atleast_1d(axes)
    for ax, analysis_label in zip(axes, top_labels):
        best_layer = int(best_layers.loc[best_layers["analysis_label"] == analysis_label, "layer"].iloc[0])
        subset = latest_preds[(latest_preds["analysis_label"] == analysis_label) & (latest_preds["layer"] == best_layer)]
        ax.scatter(subset["label"], subset["pred"], alpha=0.7)
        if len(subset) > 1:
            lo = min(subset["label"].min(), subset["pred"].min())
            hi = max(subset["label"].max(), subset["pred"].max())
            ax.plot([lo, hi], [lo, hi], linestyle="--", color="black", linewidth=1)
        ax.set_title(f"{analysis_label} | best layer L{best_layer:03d}")
        ax.set_xlabel("True label")
        ax.set_ylabel("Prediction")
    plt.tight_layout()


In [ ]:
if latest_is_eval_adoption:
    latest_runs.sort_values(["probe_control", "full_test_pearson"], ascending=[True, False])[[
        "probe_control",
        "perturbation_type",
        "layer",
        "full_test_pearson",
        "full_test_error",
        "best_val_pearson",
        "best_val_error",
    ]].head(20)
else:
    latest_runs.sort_values("full_test_pearson", ascending=False)[[
        "analysis_label",
        "layer",
        "full_test_pearson",
        "full_test_error",
        "best_val_pearson",
        "best_val_error",
    ]].head(20)


In [ ]:
runs_df

## Control Views

This section compares `NONE` and `RANDOMIZATION` control-task runs. For eval-adoption result groups, plots facet by control and color by perturbation type.


In [ ]:
classic_controls = ["NONE", "RANDOMIZATION"]
classic_df = runs_df[runs_df["control_task"].isin(classic_controls)].copy()
classic_groups = sorted(classic_df["results_group"].unique()) if not classic_df.empty else []

if classic_df.empty:
    print("No NONE/RANDOMIZATION runs found under results/.")
else:
    print("Classic control result groups:")
    for group in classic_groups:
        print(" -", group or "<root>")

classic_df.head()


In [ ]:
if not classic_df.empty:
    preferred_metrics = [
        "full_test_pearson",
        "full_test_error",
    ]
    fallback_metrics = [
        "full_test_acc",
        "full_test_f1",
    ]
    classic_metric_cols = [
        c for c in preferred_metrics if c in classic_df.columns and classic_df[c].notna().any()
    ]
    if not classic_metric_cols:
        classic_metric_cols = [
            c for c in fallback_metrics if c in classic_df.columns and classic_df[c].notna().any()
        ]
    summary_group_cols = ["results_group", "control_task"]
    if classic_df["perturbation_type"].notna().any():
        summary_group_cols.append("perturbation_type")
    classic_summary = (
        classic_df.groupby(summary_group_cols)[classic_metric_cols]
        .agg(["mean", "std", "min", "max"])
        .round(4)
    )
    display(classic_summary)


In [ ]:
if not classic_df.empty:
    latest_classic_group = classic_df.groupby("results_group")["mtime"].max().sort_values(ascending=False).index[0]
    latest_classic = classic_df[classic_df["results_group"] == latest_classic_group].copy()
    print("Latest control group:", latest_classic_group or "<root>")
    latest_classic["control_label"] = latest_classic["control_task"].str.lower()
    has_perturbations = latest_classic["perturbation_type"].notna().any()
    if "full_test_pearson" in latest_classic.columns and latest_classic["full_test_pearson"].notna().any():
        if has_perturbations:
            fig = px.line(
                latest_classic.sort_values("layer"),
                x="layer",
                y="full_test_pearson",
                color="perturbation_type",
                facet_col="control_label",
                markers=True,
                category_orders={"control_label": ["none", "randomization"]},
                title="Control-task Full-Test Pearson by Layer and Perturbation Type",
            )
            fig.for_each_annotation(lambda a: a.update(text=a.text.replace("control_label=", "control=")))
            fig.update_xaxes(title_text="Layer")
            fig.update_yaxes(title_text="Pearson correlation")
            fig.show()
        else:
            fig = px.line(
                latest_classic.sort_values("layer"),
                x="layer",
                y="full_test_pearson",
                color="control_label",
                markers=True,
                category_orders={"control_label": ["none", "randomization"]},
                title="Control-task Full-Test Pearson by Layer",
            )
            fig.update_xaxes(title_text="Layer")
            fig.update_yaxes(title_text="Pearson correlation")
            fig.show()
    else:
        metric = "full_test_acc" if "full_test_acc" in latest_classic.columns else "full_test_f1"
        if has_perturbations:
            fig = px.line(
                latest_classic.sort_values("layer"),
                x="layer",
                y=metric,
                color="perturbation_type",
                facet_col="control_label",
                markers=True,
                category_orders={"control_label": ["none", "randomization"]},
                title=f"Control-task {metric} by Layer and Perturbation Type",
            )
            fig.for_each_annotation(lambda a: a.update(text=a.text.replace("control_label=", "control=")))
            fig.update_xaxes(title_text="Layer")
            fig.update_yaxes(title_text=metric)
            fig.show()
        else:
            fig = px.line(
                latest_classic.sort_values("layer"),
                x="layer",
                y=metric,
                color="control_label",
                markers=True,
                category_orders={"control_label": ["none", "randomization"]},
                title=f"Control-task {metric} by Layer",
            )
            fig.update_xaxes(title_text="Layer")
            fig.update_yaxes(title_text=metric)
            fig.show()


In [ ]:
if not classic_df.empty:
    latest_classic_group = classic_df.groupby("results_group")["mtime"].max().sort_values(ascending=False).index[0]
    latest_classic = classic_df[classic_df["results_group"] == latest_classic_group].copy()
    latest_classic["control_label"] = latest_classic["control_task"].str.lower()
    heatmap_metric = "full_test_pearson" if "full_test_pearson" in latest_classic.columns and latest_classic["full_test_pearson"].notna().any() else "full_test_acc"
    has_perturbations = latest_classic["perturbation_type"].notna().any()

    if has_perturbations:
        fig = px.scatter(
            latest_classic,
            x="layer",
            y="perturbation_type",
            color=heatmap_metric,
            facet_col="control_label",
            category_orders={"control_label": ["none", "randomization"]},
            color_continuous_scale="RdBu" if heatmap_metric.endswith("pearson") else "Viridis",
            range_color=[-1, 1] if heatmap_metric.endswith("pearson") else None,
            title=f"Control-task Heatmap View: {heatmap_metric}",
        )
        fig.update_traces(marker={"size": 14, "symbol": "square"})
        fig.for_each_annotation(lambda a: a.update(text=a.text.replace("control_label=", "control=")))
        fig.update_xaxes(title_text="Layer")
        fig.update_yaxes(title_text="Perturbation type")
        fig.show()
    else:
        heatmap_df = latest_classic.pivot(index="control_task", columns="layer", values=heatmap_metric).sort_index()
        fig, ax = plt.subplots(figsize=(14, 3.5))
        if heatmap_metric.endswith("pearson"):
            im = ax.imshow(heatmap_df.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
            color_label = "Pearson correlation"
            title = "Classic Controls: Full-Test Pearson Heatmap"
        else:
            im = ax.imshow(heatmap_df.to_numpy(), aspect="auto", cmap="viridis", vmin=0, vmax=1)
            color_label = heatmap_metric.replace("_", " ")
            title = "Classic Controls: Full-Test Accuracy Heatmap"
        ax.set_title(title)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Control")
        ax.set_xticks(np.arange(len(heatmap_df.columns)))
        ax.set_xticklabels(heatmap_df.columns)
        ax.set_yticks(np.arange(len(heatmap_df.index)))
        ax.set_yticklabels(heatmap_df.index)
        plt.colorbar(im, ax=ax, label=color_label)
        plt.tight_layout()
